In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
 
spark = (
    SparkSession.builder
    .appName("gold-showcase")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
    .config("spark.sql.defaultCatalog", "lakehouse")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")

Spark 4.1.0


In [2]:
print(spark.table("lakehouse.cdc.bronze_customers").count())
print(spark.table("lakehouse.cdc.bronze_drivers").count())

from pyspark.sql import functions as F
spark.table("lakehouse.cdc.bronze_customers").select(
    F.countDistinct("after_id").alias("unique_after_id"),
    F.countDistinct("before_id").alias("unique_before_id"),
).show()


91
56
+---------------+----------------+
|unique_after_id|unique_before_id|
+---------------+----------------+
|             46|              12|
+---------------+----------------+



In [3]:
from pyspark.sql import functions as F, Window

bronze_df = spark.table("lakehouse.cdc.bronze_customers")

bronze_with_key = bronze_df.withColumn(
    "entity_id", F.coalesce(F.col("after_id"), F.col("before_id"))
)

w = Window.partitionBy("entity_id").orderBy(
    F.col("ts_ms").desc(), F.col("kafka_offset").desc()
)

deduped = (bronze_with_key
    .filter(F.col("op").isNotNull())
    .filter(F.col("entity_id").isNotNull())
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)

# Mitu on op='d' (kustutatud)?
deduped.groupBy("op").count().show()

+---+-----+
| op|count|
+---+-----+
|  d|   12|
|  c|   15|
|  u|   15|
|  r|    4|
+---+-----+



In [4]:
drivers = spark.read.table("lakehouse.cdc.silver_drivers").count()
zones = spark.read.table("lakehouse.taxi.gold_demand_patterns").select("pickup_zone").distinct().count()
avg_demand = spark.sql("SELECT AVG(avg_trip_count) as avg FROM lakehouse.taxi.gold_demand_patterns").collect()[0]["avg"]

print(f"Total drivers:       {drivers}")
print(f"Total zones:         {zones}")
print(f"Drivers per zone:    {drivers / zones:.4f}")
print(f"Avg demand per zone: {avg_demand:.4f}")

Total drivers:       16
Total zones:         68
Drivers per zone:    0.2353
Avg demand per zone: 16.0667


In [5]:
print(" Demand patterns overview ")
spark.sql("""
    SELECT
        pickup_zone,
        hour_of_day,
        ROUND(avg_trip_count, 2)    AS avg_trips,
        ROUND(stddev_trip_count, 2) AS std_dev,
        demand_classification
    FROM lakehouse.taxi.gold_demand_patterns
    ORDER BY avg_trips DESC
    LIMIT 10
""").show(truncate=False)

 Demand patterns overview 
+-----------------------------+-----------+---------+-------+---------------------+
|pickup_zone                  |hour_of_day|avg_trips|std_dev|demand_classification|
+-----------------------------+-----------+---------+-------+---------------------+
|Lincoln Square East          |0          |69.0     |0.0    |high demand          |
|Upper East Side South        |0          |67.0     |0.0    |high demand          |
|East Village                 |0          |64.0     |0.0    |high demand          |
|Upper West Side South        |0          |62.0     |0.0    |high demand          |
|Yorkville West               |0          |54.0     |0.0    |high demand          |
|Sutton Place/Turtle Bay North|0          |48.0     |0.0    |high demand          |
|Gramercy                     |0          |41.0     |0.0    |high demand          |
|West Village                 |0          |41.0     |0.0    |high demand          |
|Greenwich Village South      |0          |39.0  

In [6]:
print("Distribution of demand classifications")
spark.sql("""
    SELECT
        demand_classification,
        COUNT(*) AS zone_hour_combinations
    FROM lakehouse.taxi.gold_demand_patterns
    GROUP BY demand_classification
    ORDER BY zone_hour_combinations DESC
""").show()


Distribution of demand classifications
+---------------------+----------------------+
|demand_classification|zone_hour_combinations|
+---------------------+----------------------+
|               normal|                    63|
|          high demand|                    12|
+---------------------+----------------------+



In [7]:
print("Which 3 zones have the most predictable demand?")
spark.sql("""
    SELECT
        pickup_zone,
        ROUND(AVG(stddev_trip_count), 4) AS avg_std_dev,
        ROUND(AVG(avg_trip_count), 2)    AS avg_trips_per_hour,
        COUNT(*)                         AS hours_observed
    FROM lakehouse.taxi.gold_demand_patterns
    GROUP BY pickup_zone
    HAVING COUNT(*) >= 3
    ORDER BY avg_std_dev ASC
    LIMIT 3
""").show(truncate=False)

Which 3 zones have the most predictable demand?
+-----------+-----------+------------------+--------------+
|pickup_zone|avg_std_dev|avg_trips_per_hour|hours_observed|
+-----------+-----------+------------------+--------------+
+-----------+-----------+------------------+--------------+



In [8]:
print("At what hour does demand peak city-wide?")
spark.sql("""
    SELECT
        hour_of_day,
        ROUND(SUM(avg_trip_count), 0)  AS total_avg_trips_citywide,
        COUNT(DISTINCT pickup_zone)    AS zones_active
    FROM lakehouse.taxi.gold_demand_patterns
    GROUP BY hour_of_day
    ORDER BY total_avg_trips_citywide DESC
""").show(24, truncate=False)

At what hour does demand peak city-wide?
+-----------+------------------------+------------+
|hour_of_day|total_avg_trips_citywide|zones_active|
+-----------+------------------------+------------+
|0          |1197.0                  |67          |
|23         |5.0                     |5           |
|1          |3.0                     |3           |
+-----------+------------------------+------------+



In [9]:
print("Supply-demand gap — underserved zones")
spark.sql("""
    SELECT
        pickup_zone,
        hour_of_day,
        ROUND(avg_trip_count, 2)     AS avg_demand,
        ROUND(drivers_available, 2)  AS drivers_available,
        ROUND(demand_supply_gap, 2)  AS gap,
        is_underserved
    FROM lakehouse.taxi.gold_supply_demand_gap
    WHERE is_underserved = TRUE
    ORDER BY demand_supply_gap DESC
    LIMIT 15
""").show(truncate=False)

Supply-demand gap — underserved zones
+-----------------------------+-----------+----------+-----------------+-----+--------------+
|pickup_zone                  |hour_of_day|avg_demand|drivers_available|gap  |is_underserved|
+-----------------------------+-----------+----------+-----------------+-----+--------------+
|Lincoln Square East          |0          |69.0      |0.92             |68.08|true          |
|Upper East Side South        |0          |67.0      |0.9              |66.1 |true          |
|East Village                 |0          |64.0      |0.86             |63.14|true          |
|Upper West Side South        |0          |62.0      |0.83             |61.17|true          |
|Yorkville West               |0          |54.0      |0.72             |53.28|true          |
|Sutton Place/Turtle Bay North|0          |48.0      |0.64             |47.36|true          |
|West Village                 |0          |41.0      |0.55             |40.45|true          |
|Gramercy             

In [10]:
total        = spark.read.table("lakehouse.taxi.gold_demand_patterns").count()
underserved  = spark.sql("SELECT COUNT(*) as c FROM lakehouse.taxi.gold_supply_demand_gap WHERE is_underserved = TRUE").collect()[0]["c"]
total_gaps   = spark.read.table("lakehouse.taxi.gold_supply_demand_gap").count()
print(f"Total zone/hour combinations analysed : {total}")
print(f"Underserved zone/hour combinations    : {underserved} / {total_gaps}")
print(f"Underserved percentage                : {round(underserved/total_gaps*100, 1)}%")

Total zone/hour combinations analysed : 75
Underserved zone/hour combinations    : 67 / 75
Underserved percentage                : 89.3%


In [11]:
# Vaata mis op-id on Bronze-is mis EI ole Silver-is
spark.sql("""
    SELECT op, COUNT(*) as cnt
    FROM lakehouse.cdc.bronze_customers
    GROUP BY op
    ORDER BY op
""").show()

+---+---+
| op|cnt|
+---+---+
|  c| 33|
|  d| 12|
|  r| 13|
|  u| 33|
+---+---+



In [12]:
spark.sql("SELECT COUNT(*) FROM lakehouse.cdc.silver_customers").show()
spark.sql("SELECT COUNT(*) FROM lakehouse.cdc.bronze_customers").show()

+--------+
|count(1)|
+--------+
|      34|
+--------+

+--------+
|count(1)|
+--------+
|      91|
+--------+



In [13]:
spark.sql("SELECT id FROM lakehouse.cdc.silver_drivers EXCEPT SELECT id FROM postgres.public.drivers;")

{"ts": "2026-05-03 10:27:00.032", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `postgres`.`public`.`drivers` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o54.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `postgres`.`public`.`drivers` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop 

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `postgres`.`public`.`drivers` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 66;
'Except false
:- Project [id#516]
:  +- SubqueryAlias lakehouse.cdc.silver_drivers
:     +- RelationV2[id#516, name#517, email#518, country#519, last_updated_ms#520L] lakehouse.cdc.silver_drivers
+- 'Project ['id]
   +- 'UnresolvedRelation [postgres, public, drivers], [], false
